# AeroLoss AI — Model 2: YOLOv8 Blade Damage Detection Training Workflow
## Production Deep Learning Training & Validation on Google Colab (GPU Accelerated)

**Project:** AeroLoss AI (Autonomous Wind Turbine Blade Inspection & Damage Quantification)  
**Module:** Model 2 — YOLOv8 Object Detection Fine-Tuning  
**Guardrail Integration:** Couplable with Model 1 (Image Quality & Blade-Relevance Validator)  
**Dataset:** Primary Defect Benchmark (1,065 inspection images, 1,584 Pascal VOC bounding boxes)  
**Canonical Damage Classes (N=6):**
0. `corrosion` (16.2%)
1. `crack` (14.1%)
2. `craze` (16.2%)
3. `hide_craze` (21.6%)
4. `surface_injure` (26.0%)
5. `thunderstrike` (5.8%)

**Integrity Safeguard:** Leakage-free, group-aware train/val/test partition (Zero duplicate or near-duplicate group leakage across splits).

---
> **Important:** Ensure you have configured Google Colab for a GPU runtime (**Runtime > Change runtime type > T4 / A100 GPU**).

### Stage 1: Environment & Dependency Setup
Install certified dependencies for Ultralytics YOLOv8 and computer vision utilities.

In [ ]:
!pip install --quiet ultralytics==8.3.40 opencv-python pyyaml matplotlib seaborn pandas pillow

import ultralytics
ultralytics.checks()

### Stage 2: Hardware & GPU Availability Check
Verify GPU hardware, VRAM capacity, and PyTorch CUDA backend.

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("GPU Device Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    print("WARNING: GPU acceleration is NOT available. Switch to a GPU runtime before training.")

!nvidia-smi

### Stage 3: Dataset Acquisition & Path Configuration
Mount Google Drive or extract the uploaded `yolo_dataset.zip`.

In [ ]:
import os
from pathlib import Path

# Set dataset root location
DATASET_ROOT = Path("/content/yolo_dataset")

# Check for Google Drive mount
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_zip = Path("/content/drive/MyDrive/aeroloss_ai/yolo_dataset.zip")
    if drive_zip.exists() and not DATASET_ROOT.exists():
        print(f"Extracting dataset from Google Drive: {drive_zip}")
        !unzip -q "{drive_zip}" -d /content/
        print("Dataset extraction complete.")
except Exception as e:
    print(f"Google Drive check: {e}")

if not DATASET_ROOT.exists():
    print("Notice: Upload yolo_dataset.zip directly to /content/ and run: !unzip -q /content/yolo_dataset.zip -d /content/")
else:
    print(f"Verified dataset directory exists at: {DATASET_ROOT.resolve()}")

### Stage 4: Verify and Emit Colab Dataset YAML Configuration
Write the canonical 6-class dataset configuration for YOLOv8.

In [ ]:
import yaml

yaml_dict = {
    "path": str(DATASET_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "corrosion",
        1: "crack",
        2: "craze",
        3: "hide_craze",
        4: "surface_injure",
        5: "thunderstrike"
    }
}

yaml_file = DATASET_ROOT / "dataset.yaml"
with open(yaml_file, "w") as f:
    yaml.dump(yaml_dict, f, sort_keys=False)

print(f"Generated dataset.yaml at: {yaml_file}")
with open(yaml_file) as f:
    print(f.read())

### Stage 5: Dataset & Label Integrity Audit
Validate that image and label files exist in equal numbers and that all bounding box coordinates are normalized in `[0, 1]`.

In [ ]:
import glob
from collections import Counter

class_names = yaml_dict["names"]

print("=== Dataset Split Audit ===")
for split in ["train", "val", "test"]:
    img_paths = sorted(glob.glob(str(DATASET_ROOT / "images" / split / "*.jpg")))
    lbl_paths = sorted(glob.glob(str(DATASET_ROOT / "labels" / split / "*.txt")))
    print(f"[{split.upper()}] Images: {len(img_paths)} | Labels: {len(lbl_paths)}")
    assert len(img_paths) == len(lbl_paths), f"Mismatch in {split} partition!"

    box_counts = Counter()
    invalid_boxes = 0
    for lp in lbl_paths:
        with open(lp) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])
                box_counts[cls_id] += 1
                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    invalid_boxes += 1

    print(f"  Total Boxes: {sum(box_counts.values())} (Invalid: {invalid_boxes})")
    for cid in sorted(class_names.keys()):
        cname = class_names[cid]
        cnt = box_counts[cid]
        print(f"    - {cname:<16}: {cnt:>4}")

### Stage 6: Training Configuration & Hyperparameters
Configure fine-tuning hyperparameters. We use `yolov8s.pt` (Small) for optimal balance of feature representation on fine composite micro-cracks and real-time drone inference throughput.

In [ ]:
# Pretrained model checkpoint
MODEL_CHECKPOINT = "yolov8s.pt"  # Options: yolov8n.pt, yolov8s.pt, yolov8m.pt

# Hyperparameters
EPOCHS = 50
IMAGE_SIZE = 1024         # Native inspection resolution
BATCH_SIZE = 16           # Reduce to 8 if T4 GPU encounters OOM with 1024x1024
SEED = 42                 # Fully reproducible training seed
PATIENCE = 15             # Early stopping patience
PROJECT_NAME = "aeroloss_yolov8_training"
RUN_NAME = "yolov8s_leakage_free_b16_e50"

print(f"Configured: Model={MODEL_CHECKPOINT}, Epochs={EPOCHS}, ImgSz={IMAGE_SIZE}, Batch={BATCH_SIZE}, Seed={SEED}")

---
### 🛑 Stage 7: TRAINING APPROVAL GATE
**DO NOT INITIATE TRAINING WITHOUT EXPLICIT USER APPROVAL.**

Ensure that:
1. The dataset audit above verified 1,065 images and 1,584 annotations with 0 invalid boxes.
2. GPU hardware is confirmed active (`CUDA Available: True`).
3. Explicit user confirmation has been granted to proceed with training.

In [ ]:
# ==========================================================================
# UNCOMMENT AND EXECUTE ONLY AFTER EXPLICIT APPROVAL IS PROVIDED BY THE USER
# ==========================================================================
# from ultralytics import YOLO
# 
# model = YOLO(MODEL_CHECKPOINT)
# 
# training_results = model.train(
#     data=str(yaml_file),
#     epochs=EPOCHS,
#     imgsz=IMAGE_SIZE,
#     batch=BATCH_SIZE,
#     seed=SEED,
#     patience=PATIENCE,
#     optimizer="auto",
#     project=PROJECT_NAME,
#     name=RUN_NAME,
#     save=True,
#     plots=True,
#     device=0
# )
# print("Training execution completed.")

### Stage 8: Quantitative Evaluation (Validation & Held-Out Test Sets)
Compute Precision, Recall, mAP@0.5, and mAP@0.5:0.95 across both validation and unseen test sets.

In [ ]:
# best_weights = Path(f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
# if best_weights.exists():
#     trained_model = YOLO(str(best_weights))
#     
#     print("=== Held-Out Test Split Evaluation ===")
#     test_metrics = trained_model.val(data=str(yaml_file), split="test", imgsz=IMAGE_SIZE)
#     print(f"Test Overall Precision:  {test_metrics.box.mp:.4f}")
#     print(f"Test Overall Recall:     {test_metrics.box.mr:.4f}")
#     print(f"Test mAP@0.5:            {test_metrics.box.map50:.4f}")
#     print(f"Test mAP@0.5:0.95:       {test_metrics.box.map:.4f}")
#     
#     print("\nPer-Class Performance on Test Set:")
#     for cid, cname in class_names.items():
#         res = test_metrics.box.class_result(cid)
#         print(f"  {cname:<16}: Precision={res[0]:.3f}, Recall={res[1]:.3f}, mAP@0.5={res[2]:.3f}")

### Stage 9: Visualize Confusion Matrix, PR Curves & Sample Detections

In [ ]:
# from IPython.display import Image as ColabImage, display
# run_dir = Path(f"{PROJECT_NAME}/{RUN_NAME}")
# 
# artifact_images = [
#     "confusion_matrix.png",
#     "results.png",
#     "val_batch0_pred.jpg"
# ]
# for img_name in artifact_images:
#     p = run_dir / img_name
#     if p.exists():
#         print(f"\n--- Displaying {img_name} ---")
#         display(ColabImage(filename=str(p)))

### Stage 10: Model Weight Export & Google Drive Archival
Persist fine-tuned model checkpoints.

In [ ]:
# import shutil
# dest_drive_dir = Path("/content/drive/MyDrive/aeroloss_ai/models")
# dest_drive_dir.mkdir(parents=True, exist_ok=True)
# 
# shutil.copy2(best_weights, dest_drive_dir / "yolov8s_blade_damage_best.pt")
# shutil.copy2(run_dir / "weights" / "last.pt", dest_drive_dir / "yolov8s_blade_damage_last.pt")
# print(f"Saved trained model weights to Google Drive: {dest_drive_dir}")

### Stage 11: Reproducibility Notes & Known Limitations
1. **Seed Reproducibility:** Training seed fixed to `42` (`torch.manual_seed(42)`).
2. **Zero Data Leakage:** Primary inspection images partitioned using group-aware clustering; near-duplicate bursts kept atomic.
3. **Resolution Fidelity:** Training executed at native 1024x1024 resolution to preserve fine leading-edge micro-cracks.
4. **Class Imbalance:** Thunderstrike (5.8%) is naturally rarer than surface injury (26.0%). YOLOv8 focal loss and multi-scale feature pyramids appropriately balance loss gradients without artificial oversampling.